# Process RHOAI documentation with Docling

This notebook is the first step in a three-notebook flow for the RHOAI product docs RAG use case.
It reads RHOAI 3.4 product documentation PDFs from S3, converts them to structured documents
using Docling's `DocumentConverter`, chunks them with `HybridChunker`, enriches each chunk with
product metadata from the corpus manifest, and writes the resulting JSONL back to S3 for the
ingestion notebook.

Each processing step maps to a future KFP pipeline component:

| Step | KFP Component |
|------|---------------|
| PDF conversion | `docling_convert_standard` |
| Chunking | `docling_chunk` |
| Metadata enrichment | `normalize_rhoai_product_doc_chunks` |
| Artifact upload | `publish_docling_split_outputs` |

## 1. Install Docling

Docling provides PDF-to-structured-document conversion with layout analysis and table extraction.

In [ ]:
%pip install -q docling docling-core

## 2. Connect to S3 and list available PDFs

The RHOAI product documentation PDFs were uploaded to S3 during project setup (`deploy.sh`).
We use the workbench environment variables to connect and cross-reference with the corpus manifest.

In [ ]:
import json
import os
from pathlib import Path

import boto3
from botocore.config import Config

s3_endpoint = os.environ["AWS_S3_ENDPOINT"]
s3_bucket = os.environ["AWS_S3_BUCKET"]
s3_prefix = os.environ.get("RHOAI_STAGE230_PRODUCT_DOCS_PREFIX", "raw/rhoai-product-docs")

s3 = boto3.client(
    "s3",
    endpoint_url=s3_endpoint,
    aws_access_key_id=os.environ["AWS_ACCESS_KEY_ID"],
    aws_secret_access_key=os.environ["AWS_SECRET_ACCESS_KEY"],
    region_name=os.environ.get("AWS_DEFAULT_REGION", "us-east-1"),
    verify=False,
    config=Config(signature_version="s3v4"),
)

manifest = json.loads(
    Path(".stage230/data/rhoai-product-docs/metadata/rhoai-3.4-product-docs.json")
    .read_text(encoding="utf-8")
)
corpus = manifest["corpus"]

response = s3.list_objects_v2(Bucket=s3_bucket, Prefix=s3_prefix + "/")
pdf_objects = [obj for obj in response.get("Contents", []) if obj["Key"].endswith(".pdf")]

print(f"S3 bucket: {s3_bucket}")
print(f"S3 prefix: {s3_prefix}")
print(f"Found {len(pdf_objects)} PDFs in S3:\n")
for obj in pdf_objects:
    print(f"  {obj['Key']}  ({obj['Size'] / 1024:.0f} KB)")

## 3. Download PDFs from S3

Download each PDF to a local temporary directory for Docling processing.

In [ ]:
import tempfile

work_dir = Path(tempfile.mkdtemp(prefix="docling-rhoai-"))
pdf_dir = work_dir / "pdfs"
pdf_dir.mkdir()

doc_lookup = {doc["source_file"]: doc for doc in manifest["documents"]}

downloaded = []
for obj in pdf_objects:
    filename = obj["Key"].rsplit("/", 1)[-1]
    local_path = pdf_dir / filename
    s3.download_file(s3_bucket, obj["Key"], str(local_path))
    doc_meta = doc_lookup.get(filename, {})
    downloaded.append({"path": local_path, "filename": filename, "metadata": doc_meta})
    print(f"Downloaded: {filename} -> {doc_meta.get('title', 'unknown')}")

print(f"\n{len(downloaded)} PDFs ready for Docling conversion in {pdf_dir}")

## 4. Convert PDFs with Docling

Use `DocumentConverter` to convert each PDF into a structured Docling document with layout
analysis and table extraction. OCR is disabled since these are digital-native PDFs.

This step maps to the `docling_convert_standard` KFP component.

In [ ]:
from docling.document_converter import DocumentConverter, PdfFormatOption
from docling.datamodel.pipeline_options import PdfPipelineOptions
from docling.datamodel.base_models import InputFormat

pipeline_options = PdfPipelineOptions(
    do_ocr=False,
    do_table_structure=True,
)

converter = DocumentConverter(
    format_options={
        InputFormat.PDF: PdfFormatOption(pipeline_options=pipeline_options),
    }
)

converted_docs = []
for item in downloaded:
    print(f"Converting: {item['filename']} ...")
    result = converter.convert(str(item["path"]))
    converted_docs.append({
        "result": result,
        "filename": item["filename"],
        "metadata": item["metadata"],
    })
    md_text = result.document.export_to_markdown()
    print(f"  -> {len(md_text)} chars of Markdown, {len(result.document.pages)} pages")

print(f"\nConverted {len(converted_docs)} documents")

## 5. Preview converted output

Show a snippet of the Markdown output from the first document to inspect conversion quality.

In [ ]:
first_doc = converted_docs[0]
md_preview = first_doc["result"].document.export_to_markdown()
print(f"=== {first_doc['filename']} (first 2000 chars) ===\n")
print(md_preview[:2000])
print("\n...")

## 6. Chunk documents with Docling HybridChunker

Split each converted document into RAG-ready chunks using Docling's `HybridChunker`, which
combines structural boundaries with token-level limits for balanced, semantically coherent chunks.

This step maps to the `docling_chunk` KFP component.

In [ ]:
from docling.chunking import HybridChunker

chunker = HybridChunker(
    tokenizer="sentence-transformers/all-MiniLM-L6-v2",
    max_tokens=512,
    merge_peers=True,
)

all_chunks = []
for item in converted_docs:
    doc = item["result"].document
    doc_chunks = list(chunker.chunk(doc))
    for i, chunk in enumerate(doc_chunks):
        all_chunks.append({
            "chunk": chunk,
            "chunk_index": i,
            "filename": item["filename"],
            "metadata": item["metadata"],
        })
    print(f"  {item['filename']}: {len(doc_chunks)} chunks")

print(f"\nTotal: {len(all_chunks)} chunks across {len(converted_docs)} documents")

## 7. Enrich chunks with product metadata

Attach `guide_slug`, topic, product version, source URL, and corpus-level fields from the
manifest to each chunk. Topic assignment uses the manifest's `topic_rules`: for each rule,
if any of its `terms` appear in the chunk text, that topic is assigned.

This step maps to the `normalize_rhoai_product_doc_chunks` KFP component.

In [ ]:
import hashlib
import re


def assign_topic(text: str, doc_meta: dict) -> tuple[str, list[str]]:
    """Assign a topic based on manifest topic_rules matching."""
    text_lower = text.lower()
    for rule in doc_meta.get("topic_rules", []):
        matched = [t for t in rule.get("terms", []) if t.lower() in text_lower]
        if matched:
            return rule["topic"], matched
    return doc_meta.get("default_topic", "general"), []


def chunk_id(guide_slug: str, index: int) -> str:
    raw = f"{corpus['corpus']}/{guide_slug}/chunk-{index:04d}"
    return hashlib.sha256(raw.encode()).hexdigest()[:16]


records = []
for item in all_chunks:
    text = item["chunk"].text
    doc_meta = item["metadata"]
    guide_slug = doc_meta.get("guide_slug", "unknown")
    topic, matched_terms = assign_topic(text, doc_meta)

    record = {
        "id": chunk_id(guide_slug, item["chunk_index"]),
        "text": text,
        "title": doc_meta.get("title", ""),
        "guide_slug": guide_slug,
        "topic": topic,
        "matched_terms": matched_terms,
        "chunk_index": item["chunk_index"],
        "document_title": doc_meta.get("title", ""),
        "documentation_category": doc_meta.get("documentation_category", ""),
        "source_url": doc_meta.get("source_url", ""),
        "source_file": item["filename"],
        "source_format": "pdf",
        "preparation_method": "docling-standard-notebook",
        "product": corpus["product"],
        "product_version": corpus["product_version"],
        "source": corpus["source"],
        "source_authority": corpus["source_authority"],
        "tenant_id": corpus["tenant_id"],
        "version": corpus["version"],
        "version_no": corpus["version_no"],
        "corpus": corpus["corpus"],
        "document_type": corpus["document_type"],
        "language": corpus["language"],
        "access_tier": corpus["access_tier"],
    }
    records.append(record)

topics = {}
for r in records:
    topics[r["topic"]] = topics.get(r["topic"], 0) + 1

print(f"Created {len(records)} enriched chunk records\n")
print("Topic distribution:")
for topic, count in sorted(topics.items()):
    print(f"  {topic}: {count}")

## 8. Preview enriched chunks

Show a sample chunk record with its full metadata to verify the enrichment.

In [ ]:
sample = records[0]
print(f"Sample chunk (id={sample['id']}):\n")
print(f"  guide:    {sample['guide_slug']}")
print(f"  topic:    {sample['topic']}")
print(f"  title:    {sample['document_title']}")
print(f"  source:   {sample['source_url']}")
print(f"  method:   {sample['preparation_method']}")
print(f"  product:  {sample['product']} {sample['product_version']}")
print(f"\nText preview ({len(sample['text'])} chars):")
print(sample["text"][:500])

## 9. Write JSONL output to S3

Save the enriched chunks as JSONL to S3 where the ingestion notebook can pick them up.

In [ ]:
output_s3_key = "processed/rhoai-product-docs/rhoai-3.4-product-docs-docling-notebook-chunks.jsonl"

jsonl_lines = "\n".join(json.dumps(r, ensure_ascii=False) for r in records)

s3.put_object(
    Bucket=s3_bucket,
    Key=output_s3_key,
    Body=jsonl_lines.encode("utf-8"),
    ContentType="application/jsonl",
)

print(f"Wrote {len(records)} chunks to s3://{s3_bucket}/{output_s3_key}")
print(f"\nThe ingestion notebook can now load these chunks from S3.")

## 10. Also save Docling artifacts to S3

Upload the converted Markdown and full Docling JSON for each document to S3 for review
and downstream use.

This step maps to the `publish_docling_split_outputs` KFP component.

In [ ]:
artifact_prefix = "processed/rhoai-product-docs/docling-artifacts"

for item in converted_docs:
    slug = item["metadata"].get("guide_slug", item["filename"].replace(".pdf", ""))

    md_content = item["result"].document.export_to_markdown()
    md_key = f"{artifact_prefix}/{slug}.md"
    s3.put_object(Bucket=s3_bucket, Key=md_key, Body=md_content.encode("utf-8"))

    json_content = json.dumps(item["result"].document.export_to_dict(), ensure_ascii=False, indent=2)
    json_key = f"{artifact_prefix}/{slug}.json"
    s3.put_object(Bucket=s3_bucket, Key=json_key, Body=json_content.encode("utf-8"))

    print(f"  {slug}: Markdown ({len(md_content)} chars) + Docling JSON")

print(f"\nDocling artifacts saved to s3://{s3_bucket}/{artifact_prefix}/")

## Summary

This notebook completed the data preparation stage:

1. Downloaded RHOAI 3.4 product documentation PDFs from S3
2. Converted them to structured documents using Docling `DocumentConverter`
3. Chunked them with `HybridChunker` (512-token semantic chunks)
4. Enriched each chunk with product metadata and topic classification from the manifest
5. Wrote the enriched chunks as JSONL to S3
6. Uploaded Docling Markdown and JSON artifacts for review

**Next:** Open `Ingestion_pipeline_rhoai_docs.ipynb` to load these chunks into the
Llama Stack vector store via OGX.

In [ ]:
import shutil
shutil.rmtree(work_dir, ignore_errors=True)
print("Temporary files cleaned up")
print(f"\nNext step: open Ingestion_pipeline_rhoai_docs.ipynb to load these chunks into the Llama Stack vector store")